# Molecular Informatics Pipeline - Exploratory Analysis

This notebook provides exploratory analysis of the molecular dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors

sns.set_style('whitegrid')
%matplotlib inline

## Load Data

In [ ]:
# Load raw data
raw_df = pd.read_csv('../data/raw/molecules.csv')
print(f"Raw data shape: {raw_df.shape}")
raw_df.head()

## Data Summary

In [ ]:
# Summary statistics
print("\nSummary Statistics:")
raw_df.describe()

In [ ]:
# Missing values
print("\nMissing Values:")
raw_df.isnull().sum()

## Molecular Property Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Molecular weight
axes[0, 0].hist(raw_df['molecular_weight'].dropna(), bins=20, edgecolor='black')
axes[0, 0].set_title('Molecular Weight Distribution')
axes[0, 0].set_xlabel('MW (g/mol)')

# XLogP
axes[0, 1].hist(raw_df['xlogp'].dropna(), bins=20, edgecolor='black', color='orange')
axes[0, 1].set_title('XLogP Distribution')
axes[0, 1].set_xlabel('XLogP')

# TPSA
axes[0, 2].hist(raw_df['tpsa'].dropna(), bins=20, edgecolor='black', color='green')
axes[0, 2].set_title('TPSA Distribution')
axes[0, 2].set_xlabel('TPSA (Ų)')

# HBD count
axes[1, 0].hist(raw_df['hbd_count'].dropna(), bins=10, edgecolor='black', color='red')
axes[1, 0].set_title('H-Bond Donor Count')
axes[1, 0].set_xlabel('HBD Count')

# HBA count
axes[1, 1].hist(raw_df['hba_count'].dropna(), bins=10, edgecolor='black', color='purple')
axes[1, 1].set_title('H-Bond Acceptor Count')
axes[1, 1].set_xlabel('HBA Count')

# MW vs LogP scatter
axes[1, 2].scatter(raw_df['molecular_weight'], raw_df['xlogp'])
axes[1, 2].set_title('MW vs XLogP')
axes[1, 2].set_xlabel('MW (g/mol)')
axes[1, 2].set_ylabel('XLogP')

plt.tight_layout()
plt.show()

## Visualize Molecular Structures

In [ ]:
# Display first 6 molecules
mols = []
legends = []

for idx, row in raw_df.head(6).iterrows():
    if pd.notna(row['smiles']):
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol is not None:
            mols.append(mol)
            legends.append(row['name'])

if mols:
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=legends)
    display(img)

## Load Processed Features

In [ ]:
# Load processed data
try:
    processed_df = pd.read_csv('../data/processed/processed.csv')
    print(f"Processed data shape: {processed_df.shape}")
    print(f"Number of features: {len([col for col in processed_df.columns if col.startswith('morgan_') or col.startswith('maccs_')])}")
except FileNotFoundError:
    print("Processed data not found. Run featurization first.")

## Feature Correlation

In [ ]:
if 'processed_df' in locals():
    descriptor_cols = ['MW', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 
                       'NumRotatableBonds', 'NumAromaticRings', 'NumAliphaticRings']
    
    corr_matrix = processed_df[descriptor_cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Descriptor Correlation Matrix')
    plt.tight_layout()
    plt.show()

## Model Performance

In [ ]:
# Load training metrics
try:
    metrics_df = pd.read_csv('../reports/training_metrics.csv')
    print("\nModel Performance Metrics:")
    display(metrics_df)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    metrics_to_plot = ['Test MSE', 'Test MAE', 'Test R2']
    for idx, metric in enumerate(metrics_to_plot):
        axes[idx].bar(metrics_df['Model'], metrics_df[metric])
        axes[idx].set_title(metric)
        axes[idx].set_ylabel(metric)
        axes[idx].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Training metrics not found. Run training first.")

## Conclusion

This notebook provides a basic exploratory analysis of the molecular dataset. Key observations:

1. Dataset contains diverse molecular properties
2. Features are computed using RDKit
3. Multiple models are trained for property prediction
4. Both linear and nonlinear models are compared

For further analysis, consider:
- Feature selection techniques
- Dimensionality reduction (PCA, t-SNE)
- Hyperparameter optimization
- External validation on independent test set